# 用 Sciverse 做多模态图表检索 Demo

> 根据自然语言描述检索论文图表，结合多模态模型分析图表内容

---

**Sciverse Cookbook** | [文档首页](https://sciverse.space/docs#cookbook) | [申请 API Key](https://sciverse.space/docs#auth)

## 前置准备

1. 在 [Sciverse 控制台](https://sciverse.space/) 申请 API Token
2. 安装依赖：`pip install httpx anthropic`
3. 将下方 `sv-...` 替换为你的真实 Token


In [ ]:
# 安装依赖（Colab 环境）
!pip install -q httpx anthropic


## Step 1: 检索包含目标图表的文献

用自然语言描述检索相关文献


In [ ]:
import httpx, re

BASE = "https://api.sciverse.space"
HEADERS = {"Authorization": "Bearer sv-..."}

async def search_figures(description: str):
    async with httpx.AsyncClient() as client:
        resp = await client.post(
            f"{BASE}/agentic-search",
            headers=HEADERS,
            json={"query": f"figure table {description}", "top_k": 10}
        )
        return resp.json()["hits"]

hits = await search_figures("AlphaFold2 experimental structure comparison GDT-TS")
print(f"Found {len(hits)} relevant documents")

## Step 2: 定位并下载图表

读取全文找到图表路径，调用 resource 下载


In [ ]:
from pathlib import Path

async def get_figures_from_doc(doc_id: str):
    async with httpx.AsyncClient() as client:
        resp = await client.get(
            f"{BASE}/content",
            headers=HEADERS,
            params={"doc_id": doc_id, "offset": 0, "limit": 4096}
        )
        content = resp.json()["content"]
        figure_paths = re.findall(r'!\[.*?\]\((.*?)\)', content)
        return figure_paths

async def download_figure(path: str, save_dir: str = "./figures"):
    Path(save_dir).mkdir(exist_ok=True)
    async with httpx.AsyncClient() as client:
        resp = await client.get(
            f"{BASE}/resource",
            headers=HEADERS,
            params={"path": path}
        )
        local = f"{save_dir}/{path.split('/')[-1]}"
        Path(local).write_bytes(resp.content)
        return local

for hit in hits[:3]:
    paths = await get_figures_from_doc(hit["doc_id"])
    for p in paths[:2]:
        local = await download_figure(p)
        print(f"Downloaded: {local}")

## Step 3: 多模态分析图表内容

用多模态 LLM 提取图表中的关键数据


In [ ]:
import base64
from anthropic import Anthropic

client = Anthropic()

async def analyze_figure(image_path: str, question: str):
    with open(image_path, "rb") as f:
        img_data = base64.b64encode(f.read()).decode()

    msg = client.messages.create(
        model="claude-opus-4-7",
        max_tokens=2048,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image", "source": {
                    "type": "base64",
                    "media_type": "image/png",
                    "data": img_data
                }},
                {"type": "text", "text": question}
            ]
        }]
    )
    return msg.content[0].text

analysis = await analyze_figure(
    "./figures/f2.png",
    "请描述这张图表中 GDT-TS 分布情况，提取关键数值和结论。"
)
print(analysis)

---

## 下一步

- 📖 [查看完整 API 文档](https://sciverse.space/docs#sciverse/api)
- 🔑 [申请 / 管理 API Token](https://sciverse.space/)
- 📚 [浏览更多 Cookbook 案例](https://sciverse.space/docs#cookbook)
- 💬 [加入开发者社群](https://sciverse.space/)
